# EcoSort — Train YOLOv8 on TrashNet

Trains a waste-image classifier (6 classes) and downloads the weights.

**Before running:** Runtime → Change runtime type → **GPU (T4)**. Then Runtime → Run all.

In [ ]:
!pip install -q ultralytics

## 1. Download TrashNet and build train/val folders

In [ ]:
import random, shutil, urllib.request, zipfile
from pathlib import Path

CLASSES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
ZIP_URL = "https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip"

# Download original TrashNet resized zip (the HF `datasets` loader is broken for this set)
zip_path = Path("dataset-resized.zip")
if not zip_path.exists():
    print("Downloading TrashNet...")
    urllib.request.urlretrieve(ZIP_URL, zip_path)
with zipfile.ZipFile(zip_path) as z:
    z.extractall("_extract")
src = Path("_extract/dataset-resized")

root = Path("trashnet")
random.seed(42)
for cls in CLASSES:
    imgs = sorted((src / cls).glob("*.jpg"))
    random.shuffle(imgs)
    split = int(0.8 * len(imgs))
    for phase, group in {"train": imgs[:split], "val": imgs[split:]}.items():
        d = root / phase / cls
        d.mkdir(parents=True, exist_ok=True)
        for p in group:
            shutil.copy(p, d / p.name)
print("Dataset ready. Classes:", CLASSES)

## 2. Train

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-cls.pt")  # smallest classifier, ~3 MB trained
results = model.train(data="trashnet", epochs=20, imgsz=224, batch=64, seed=42)

## 3. Validation accuracy (cite these real numbers)

In [ ]:
metrics = model.val()
print(f"Top-1 accuracy: {metrics.top1:.3f}")
print(f"Top-5 accuracy: {metrics.top5:.3f}")

## 4. Download the trained weights

Save the downloaded `ecosort.pt` into the repo at `models/ecosort.pt`.

In [ ]:
import shutil
from google.colab import files

best = Path(results.save_dir) / "weights" / "best.pt"
shutil.copy(best, "ecosort.pt")
print("Size:", round(Path("ecosort.pt").stat().st_size / 1e6, 1), "MB")
files.download("ecosort.pt")